In [9]:
import pandas as pd

# Carregar test set
test = pd.read_csv(
    "/home/eduardo-da-paz/Documents/master/tablerag-slm/data/WikiTableQuestions/data/random-split-1-dev.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="skip" 
    )
test.head()

,id,utterance,context,targetValue
0,nt-2,which team won previous to crettyard?,csv/204-csv/772.csv,Wolfe Tones
1,nt-3,how many more passengers flew to los angeles t...,csv/203-csv/515.csv,"12,467"
2,nt-8,after winning on four credits with a full hous...,csv/203-csv/564.csv,32
3,nt-9,which players played the same position as ardo...,csv/203-csv/116.csv,Siim Ennemuist|Andri Aganits
4,nt-14,which athlete was from south korea after the y...,csv/203-csv/104.csv,Kim Yu-na


In [11]:
import os

# Testar se consegue carregar uma tabela
table_path = os.path.join(
    "/home/eduardo-da-paz/Documents/master/tablerag-slm/data/raw/WikiTableQuestions",
    "csv/203-csv/733.csv"
)   

table = pd.read_csv(table_path)
print(table.head())
print(f"Shape: {table.shape}")

   Rank                   Cyclist                Team             Time  \
0     1  Alejandro Valverde (ESP)    Caisse d'Epargne  5h 29' 10\",40"   
1     2   Alexandr Kolobnev (RUS)  Team CSC Saxo Bank             s.t.   
2     3     Davide Rebellin (ITA)        Gerolsteiner             s.t.   
3     4       Paolo Bettini (ITA)          Quick Step             s.t.   
4     5   Franco Pellizotti (ITA)            Liquigas             s.t.   

   UCI ProTour\nPoints  
0                  NaN  
1                 30.0  
2                 25.0  
3                 20.0  
4                 15.0  
Shape: (10, 5)


In [16]:

# ─── 1. CARREGAR O TEST SET ───────────────────────────────────────────
test = pd.read_csv(
    "/home/eduardo-da-paz/Documents/master/tablerag-slm/data/raw/WikiTableQuestions/data/pristine-unseen-tables.tsv",
    sep="\t", header=0
)
print("=== 1. SHAPE E COLUNAS ===")
print(f"Total exemplos: {len(test)}")
print(f"Colunas: {test.columns.tolist()}")

# ─── 2. VALORES NULOS ────────────────────────────────────────────────
print("\n=== 2. VALORES NULOS ===")
print(test.isnull().sum())

# ─── 3. EXEMPLO COMPLETO ─────────────────────────────────────────────
print("\n=== 3. EXEMPLO COMPLETO ===")
print(test.iloc[0].to_dict())

# ─── 4. DISTRIBUIÇÃO DAS RESPOSTAS ───────────────────────────────────
print("\n=== 4. TIPOS DE RESPOSTA ===")
print(f"Respostas únicas: {test['targetValue'].nunique()}")
print(f"Respostas com múltiplos valores (|): {test['targetValue'].str.contains('|', regex=False).sum()}")

# ─── 5. TABELAS ACESSÍVEIS ───────────────────────────────────────────
print("\n=== 5. TABELAS ACESSÍVEIS ===")
base_dir = "/home/eduardo-da-paz/Documents/master/tablerag-slm/data/raw/WikiTableQuestions"
missing = 0
for ctx in test["context"]:
    path = os.path.join(base_dir, ctx)
    if not os.path.exists(path):
        missing += 1

print(f"Tabelas encontradas: {len(test) - missing}/{len(test)}")
print(f"Tabelas faltando: {missing}")

# ─── 6. TABELAS ÚNICAS ───────────────────────────────────────────────
print("\n=== 6. TABELAS ÚNICAS ===")
unique_tables = test["context"].nunique()
print(f"Total de tabelas únicas: {unique_tables}")
print(f"Média de perguntas por tabela: {len(test) / unique_tables:.1f}")

# ─── 7. CARREGAR UMA TABELA DE EXEMPLO ──────────────────────────────
print("\n=== 7. EXEMPLO DE TABELA ===")
sample_ctx = test.iloc[0]["context"]
table = pd.read_csv(os.path.join(base_dir, sample_ctx))
print(f"Tabela: {sample_ctx}")
print(f"Shape: {table.shape}")
print(table.head(3))

# ─── 8. TAMANHO DAS TABELAS ──────────────────────────────────────────
print("\n=== 8. ESTATÍSTICAS DE TAMANHO DAS TABELAS ===")
shapes = []
for ctx in test["context"].unique()[:100]:  # amostra de 100
    path = os.path.join(base_dir, ctx)
    if os.path.exists(path):
        df = pd.read_csv(path, on_bad_lines="skip")
        shapes.append({"rows": df.shape[0], "cols": df.shape[1]})

shapes_df = pd.DataFrame(shapes)
print(f"Linhas — min: {shapes_df['rows'].min()}, max: {shapes_df['rows'].max()}, média: {shapes_df['rows'].mean():.1f}")
print(f"Colunas — min: {shapes_df['cols'].min()}, max: {shapes_df['cols'].max()}, média: {shapes_df['cols'].mean():.1f}")

# ─── 9. COMPRIMENTO DAS PERGUNTAS ────────────────────────────────────
print("\n=== 9. COMPRIMENTO DAS PERGUNTAS ===")
test["q_len"] = test["utterance"].str.split().str.len()
print(f"Palavras por pergunta — min: {test['q_len'].min()}, max: {test['q_len'].max()}, média: {test['q_len'].mean():.1f}")

# ─── 10. SUBCONJUNTO DE 200 ──────────────────────────────────────────
print("\n=== 10. AMOSTRA DE 200 PERGUNTAS ===")
subset = test.sample(n=200, random_state=42)
subset.to_json("/home/eduardo-da-paz/Documents/master/tablerag-slm/data/processed/test_200.jsonl", orient="records", lines=True)
print(f"Subconjunto salvo: {len(subset)} exemplos")
print(f"Tabelas únicas no subconjunto: {subset['context'].nunique()}")

=== 1. SHAPE E COLUNAS ===
Total exemplos: 4344
Colunas: ['id', 'utterance', 'context', 'targetValue']

=== 2. VALORES NULOS ===
id             0
utterance      0
context        0
targetValue    0
dtype: int64

=== 3. EXEMPLO COMPLETO ===
{'id': 'nu-0', 'utterance': 'which country had the most cyclists finish within the top 10?', 'context': 'csv/203-csv/733.csv', 'targetValue': 'Italy'}

=== 4. TIPOS DE RESPOSTA ===
Respostas únicas: 2012
Respostas com múltiplos valores (|): 115

=== 5. TABELAS ACESSÍVEIS ===
Tabelas encontradas: 4344/4344
Tabelas faltando: 0

=== 6. TABELAS ÚNICAS ===
Total de tabelas únicas: 421
Média de perguntas por tabela: 10.3

=== 7. EXEMPLO DE TABELA ===
Tabela: csv/203-csv/733.csv
Shape: (10, 5)
   Rank                   Cyclist                Team             Time  \
0     1  Alejandro Valverde (ESP)    Caisse d'Epargne  5h 29' 10\",40"   
1     2   Alexandr Kolobnev (RUS)  Team CSC Saxo Bank             s.t.   
2     3     Davide Rebellin (ITA)        Gerols

In [4]:
import torch

print(f"Torch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Torch Version: 2.11.0+cu130
CUDA Available: True


In [2]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


In [2]:
import json

# 1. Lendo apenas a primeira entrada do dev.json (Perguntas)
print("=== PRIMEIRA ENTRADA DO DEV.JSON ===")
with open('/home/eduardo-da-paz/Documents/master/tablerag-slm/data/raw/OTT-QA/dev.json', 'r', encoding='utf-8') as f:
    dev_data = json.load(f)
    
# Como dev.json é uma lista, pegamos o índice [0]
primeira_pergunta = dev_data[0]
print(json.dumps(primeira_pergunta, indent=4, ensure_ascii=False))

print("\n" + "="*50 + "\n")

# 2. Lendo apenas a primeira entrada do traindev_tables.json (Tabelas)
print("=== PRIMEIRA ENTRADA DO TRAINDEV_TABLES.JSON ===")
with open('/home/eduardo-da-paz/Documents/master/tablerag-slm/data/raw/OTT-QA/traindev_tables.json', 'r', encoding='utf-8') as f:
    # Nota: Este arquivo é grande. Carregar com json.load pode demorar alguns segundos.
    tables_data = json.load(f)
    
# Como traindev_tables.json é um dicionário (chave: id da tabela), 
# pegamos a primeira chave disponível para exibir
primeira_chave = list(tables_data.keys())[0]
primeira_tabela = tables_data[primeira_chave]

print(f"ID da Tabela (Chave): {primeira_chave}\n")
print(json.dumps(primeira_tabela, indent=4, ensure_ascii=False))


=== PRIMEIRA ENTRADA DO DEV.JSON ===
{
    "question_id": "2b6359edb1b352c3",
    "question": "Who created the series in which the character of Robert , played by actor Nonso Anozie , appeared ?",
    "table_id": "Nonso_Anozie_1",
    "answer-text": "Lynda La Plante",
    "question_postag": "WP VBD DT NN IN WDT DT NN IN NNP , VBN IN NN NNP NNP , VBD ."
}


=== PRIMEIRA ENTRADA DO TRAINDEV_TABLES.JSON ===
ID da Tabela (Chave): Serbia_at_the_European_Athletics_Championships_2

{
    "url": "https://en.wikipedia.org/wiki/Serbia_at_the_European_Athletics_Championships",
    "title": "Serbia at the European Athletics Championships",
    "header": [
        [
            "Medal",
            []
        ],
        [
            "Name",
            []
        ],
        [
            "Event",
            []
        ],
        [
            "Championship",
            []
        ]
    ],
    "data": [
        [
            [
                "Gold",
                []
            ],
            